In [ ]:
# -------------------------------------------------------------
# Google Colab Single-Cell Runner (Milestone 2 - Light Theme)
# -------------------------------------------------------------

# 1. Install required packages
!pip install -q streamlit pyjwt bcrypt scikit-learn joblib pandas numpy plotly pyngrok transformers bitsandbytes accelerate sentencepiece protobuf

import os
import json
import time
import subprocess

# 2. Write db.py
with open("db.py", "w", encoding="utf-8") as f:
    f.write('''import sqlite3
import bcrypt
import datetime
import time

DB_NAME = "milestone2.db"

def get_connection():
    return sqlite3.connect(DB_NAME, check_same_thread=False)

def init_db():
    with get_connection() as conn:
        c = conn.cursor()
        c.execute("""
            CREATE TABLE IF NOT EXISTS users (
                email TEXT PRIMARY KEY,
                username TEXT UNIQUE NOT NULL,
                password_hash TEXT NOT NULL,
                role TEXT NOT NULL DEFAULT 'Viewer',
                failed_attempts INTEGER DEFAULT 0,
                lock_until REAL DEFAULT 0.0,
                account_status TEXT DEFAULT 'active',
                security_question TEXT NOT NULL,
                security_answer_hash TEXT NOT NULL,
                created_at TEXT NOT NULL,
                otp_resend_count INTEGER DEFAULT 0,
                last_otp_sent_at REAL DEFAULT 0.0
            )
        """)
        c.execute("""
            CREATE TABLE IF NOT EXISTS password_history (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                email TEXT NOT NULL,
                password_hash TEXT NOT NULL,
                set_at TEXT NOT NULL,
                FOREIGN KEY(email) REFERENCES users(email) ON DELETE CASCADE
            )
        """)
        c.execute("""
            CREATE TABLE IF NOT EXISTS ml_models (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                agent_name TEXT NOT NULL,
                algorithm TEXT NOT NULL,
                accuracy REAL,
                R2 REAL,
                RMSE REAL,
                ROC_AUC REAL,
                model_path TEXT NOT NULL,
                created_at TEXT NOT NULL
            )
        """)
        conn.commit()

    preseed_users = [
        ("admin@portal.com", "Admin_Portal", "AdminPassword@2026", "What is your favourite city?", "bengaluru", "Admin"),
        ("springboardmentor018@gmail.com", "Mentor_018", "Welcome@123", "What is your favourite city?", "bengaluru", "Logistics Manager"),
        ("springboardmentor038@gmail.com", "Mentor_038", "Welcome@123", "What is your pet name?", "shadow", "Viewer")
    ]
    for email, username, pwd, sq, sa, role in preseed_users:
        if not check_user_exists(email) and not check_username_exists(username):
            register_user(email, username, pwd, sq, sa, role)

def _get_timestamp():
    return datetime.datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")

def hash_text(text):
    return bcrypt.hashpw(text.encode('utf-8'), bcrypt.gensalt()).decode('utf-8')

def check_hash(text, hashed):
    if not hashed: return False
    try: return bcrypt.checkpw(text.encode('utf-8'), hashed.encode('utf-8'))
    except: return False

def check_user_exists(email):
    with get_connection() as conn:
        c = conn.cursor()
        c.execute("SELECT 1 FROM users WHERE LOWER(email) = ?", (email.lower().strip(),))
        return c.fetchone() is not None

def check_username_exists(username):
    with get_connection() as conn:
        c = conn.cursor()
        c.execute("SELECT 1 FROM users WHERE LOWER(username) = ?", (username.lower().strip(),))
        return c.fetchone() is not None

def check_lockout_status(email):
    email_clean = email.lower().strip()
    with get_connection() as conn:
        c = conn.cursor()
        c.execute("SELECT account_status, lock_until FROM users WHERE email = ?", (email_clean,))
        row = c.fetchone()
    if not row: return False, "active", 0
    status, lock_until = row
    current_time = time.time()
    if status == 'perm_locked': return True, 'perm_locked', 0
    if lock_until and current_time < lock_until:
        return True, 'temp_locked', int(lock_until - current_time)
    return False, 'active', 0

def handle_failed_login(email):
    email_clean = email.lower().strip()
    current_time = time.time()
    with get_connection() as conn:
        c = conn.cursor()
        c.execute("SELECT failed_attempts, account_status FROM users WHERE email = ?", (email_clean,))
        row = c.fetchone()
        if not row: return False, "active", 0
        attempts, status = row
        if status == 'perm_locked': return True, 'perm_locked', 0
        new_attempts = attempts + 1
        new_status = 'active'
        lock_until = 0.0
        if new_attempts == 3:
            lock_until = current_time + 300
            new_status = 'temp_locked'
        elif new_attempts == 4:
            lock_until = current_time + 900
            new_status = 'temp_locked'
        elif new_attempts >= 5:
            new_status = 'perm_locked'
        c.execute("""
            UPDATE users SET failed_attempts = ?, lock_until = ?, account_status = ? WHERE email = ?
        """, (new_attempts, lock_until, new_status, email_clean))
        conn.commit()
    return check_lockout_status(email_clean)

def reset_login_attempts(email):
    email_clean = email.lower().strip()
    with get_connection() as conn:
        c = conn.cursor()
        c.execute("""
            UPDATE users SET failed_attempts = 0, lock_until = 0.0, account_status = 'active'
            WHERE email = ? AND account_status != 'perm_locked'
        """, (email_clean,))
        conn.commit()

def check_otp_resend_cooldown(email):
    email_clean = email.lower().strip()
    current_time = time.time()
    with get_connection() as conn:
        c = conn.cursor()
        c.execute("SELECT otp_resend_count, last_otp_sent_at FROM users WHERE email = ?", (email_clean,))
        row = c.fetchone()
    if not row: return True, 0, 0
    resend_count, last_sent = row
    if resend_count == 0 or not last_sent: return True, 0, 0
    if resend_count == 1: cooldown = 60
    elif resend_count == 2: cooldown = 180
    elif resend_count == 3: cooldown = 300
    else: cooldown = 3600
    elapsed = current_time - last_sent
    if elapsed < cooldown: return False, int(cooldown - elapsed), resend_count
    return True, 0, resend_count

def record_otp_sent(email):
    email_clean = email.lower().strip()
    current_time = time.time()
    with get_connection() as conn:
        c = conn.cursor()
        c.execute("""
            UPDATE users SET otp_resend_count = otp_resend_count + 1, last_otp_sent_at = ? WHERE email = ?
        """, (current_time, email_clean))
        conn.commit()

def reset_otp_cooldown(email):
    email_clean = email.lower().strip()
    with get_connection() as conn:
        c = conn.cursor()
        c.execute("UPDATE users SET otp_resend_count = 0, last_otp_sent_at = 0.0 WHERE email = ?", (email_clean,))
        conn.commit()

def register_user(email, username, password, security_question, security_answer, role='Viewer'):
    email_clean = email.lower().strip()
    username_clean = username.strip()
    sa_clean = security_answer.lower().strip()
    pwd_hashed = hash_text(password)
    sa_hashed = hash_text(sa_clean)
    now = _get_timestamp()
    with get_connection() as conn:
        c = conn.cursor()
        try:
            c.execute("""
                INSERT INTO users (email, username, password_hash, role, security_question, security_answer_hash, created_at)
                VALUES (?, ?, ?, ?, ?, ?, ?)
            """, (email_clean, username_clean, pwd_hashed, role, security_question, sa_hashed, now))
            c.execute("INSERT INTO password_history (email, password_hash, set_at) VALUES (?, ?, ?)", (email_clean, pwd_hashed, now))
            conn.commit()
            return True
        except:
            return False

def authenticate_user(email, password):
    email_clean = email.lower().strip()
    with get_connection() as conn:
        c = conn.cursor()
        c.execute("SELECT password_hash, role, account_status FROM users WHERE email = ?", (email_clean,))
        row = c.fetchone()
    if row:
        stored_hash, role, status = row
        if status == 'perm_locked': return False
        if check_hash(password, stored_hash):
            reset_login_attempts(email_clean)
            return True
    return False

def get_user_by_email(email):
    with get_connection() as conn:
        c = conn.cursor()
        c.execute("SELECT username, role, security_question, security_answer_hash, account_status FROM users WHERE email = ?", (email.lower().strip(),))
        return c.fetchone()

def check_password_reused(email, new_password):
    email_clean = email.lower().strip()
    with get_connection() as conn:
        c = conn.cursor()
        c.execute("SELECT password_hash FROM password_history WHERE email = ?", (email_clean,))
        history = c.fetchall()
    for (stored_hash,) in history:
        if check_hash(new_password, stored_hash): return True
    return False

def update_password(email, new_password):
    email_clean = email.lower().strip()
    pwd_hashed = hash_text(new_password)
    now = _get_timestamp()
    with get_connection() as conn:
        c = conn.cursor()
        c.execute("UPDATE users SET password_hash = ? WHERE email = ?", (pwd_hashed, email_clean))
        c.execute("INSERT INTO password_history (email, password_hash, set_at) VALUES (?, ?, ?)", (email_clean, pwd_hashed, now))
        conn.commit()

def get_all_users():
    with get_connection() as conn:
        c = conn.cursor()
        c.execute("SELECT username, email, role, failed_attempts, account_status, created_at FROM users")
        return c.fetchall()

def delete_user(email):
    email_clean = email.lower().strip()
    with get_connection() as conn:
        c = conn.cursor()
        c.execute("DELETE FROM users WHERE email = ?", (email_clean,))
        c.execute("DELETE FROM password_history WHERE email = ?", (email_clean,))
        conn.commit()
        return True

def unlock_user(email):
    email_clean = email.lower().strip()
    with get_connection() as conn:
        c = conn.cursor()
        c.execute("UPDATE users SET failed_attempts = 0, lock_until = 0.0, account_status = 'active' WHERE email = ?", (email_clean,))
        conn.commit()
        return True

def update_user_role(email, new_role):
    email_clean = email.lower().strip()
    with get_connection() as conn:
        c = conn.cursor()
        c.execute("UPDATE users SET role = ? WHERE email = ?", (new_role, email_clean))
        conn.commit()
        return True

def log_trained_model(agent_name, algorithm, accuracy, R2, RMSE, ROC_AUC, model_path):
    now = _get_timestamp()
    with get_connection() as conn:
        c = conn.cursor()
        c.execute("""
            INSERT INTO ml_models (agent_name, algorithm, accuracy, R2, RMSE, ROC_AUC, model_path, created_at)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?)
        """, (agent_name, algorithm, accuracy, R2, RMSE, ROC_AUC, model_path, now))
        conn.commit()

def get_trained_models():
    with get_connection() as conn:
        c = conn.cursor()
        c.execute("SELECT agent_name, algorithm, accuracy, R2, RMSE, ROC_AUC, model_path, created_at FROM ml_models ORDER BY created_at DESC")
        return c.fetchall()
''')

# 3. Write ui_theme.py
with open("ui_theme.py", "w", encoding="utf-8") as f:
    f.write('''import streamlit as st

# Modern Neo-Brutalist Light Palette
COLORS = {
    "bg_main": "#f1f5f9",        # Light Slate gray background
    "bg_sidebar": "#ffffff",     # White Sidebar
    "bg_card": "#ffffff",        # White Cards
    "bg_card_alt": "#f8fafc",    # Slate 50 for alternate items
    "text_main": "#0f172a",      # Slate 900 dark text
    "text_heading": "#0f172a",   # Slate 900 dark headings
    "text_muted": "#475569",     # Slate 600 muted text
    "accent": "#6366f1",         # Indigo accent
    "accent_hover": "#4f46e5",   # Darker indigo hover
    "accent_text": "#ffffff",    # White text on buttons
    "border": "#0f172a",         # Bold dark borders
    "border_light": "#cbd5e1",   # Light slate borders
    "success": "#10b981",        # Emerald Green
    "danger": "#ef4444",         # Rose Red
    "cyan": "#06b6d4"            # Cyan highlight shadow
}

def inject_theme_css():
    st.markdown(f"""
    <style>
        @import url('https://fonts.googleapis.com/css2?family=Poppins:wght@400;500;600;700&family=Inter:wght@300;400;500;600;700&display=swap');

        html, body, .stApp {{
            background: {COLORS['bg_main']} !important;
            font-family: 'Inter', sans-serif !important;
            color: {COLORS['text_main']} !important;
        }}

        footer, div[data-testid="stDecoration"] {{
            visibility: hidden !important;
            display: none !important;
        }}
        header {{
            background: transparent !important;
            z-index: 999999 !important;
        }}

        button[kind="header"], div[data-testid="stSidebarCollapsedControl"] button {{
            background-color: {COLORS['accent']} !important;
            border: 2px solid {COLORS['border']} !important;
            border-radius: 8px !important;
            padding: 6px !important;
            margin: 8px !important;
            box-shadow: 3px 3px 0px {COLORS['cyan']} !important;
        }}

        .block-container {{
            padding: 2rem 2.5rem !important;
            max-width: 1200px;
        }}

        h1, h2, h3, h4, h5 {{
            font-family: 'Poppins', sans-serif !important;
            color: {COLORS['text_heading']} !important;
            font-weight: 700 !important;
        }}

        label p {{
            font-weight: 600 !important;
            color: {COLORS['text_heading']} !important;
        }}

        div[data-baseweb="input"], div[data-baseweb="select"] {{
            background-color: {COLORS['bg_card_alt']} !important;
            border: 2px solid {COLORS['border']} !important;
            border-radius: 10px !important;
            padding: 2px 4px !important;
        }}

        input, div[data-baseweb="select"] span {{
            color: {COLORS['text_main']} !important;
            -webkit-text-fill-color: {COLORS['text_main']} !important;
            font-weight: 500 !important;
        }}

        div[data-testid="stButton"] button {{
            background-color: {COLORS['accent']} !important;
            color: {COLORS['accent_text']} !important;
            border: 2px solid {COLORS['border']} !important;
            border-radius: 10px !important;
            font-family: 'Poppins', sans-serif !important;
            font-weight: 700 !important;
            box-shadow: 4px 4px 0px {COLORS['cyan']} !important;
            width: 100%;
        }}

        section[data-testid="stSidebar"] {{
            background: {COLORS['bg_sidebar']} !important;
            border-right: 2px solid {COLORS['border']} !important;
        }}

        .nb-card {{
            background-color: {COLORS['bg_card']} !important;
            border: 2px solid {COLORS['border']} !important;
            border-radius: 14px !important;
            padding: 24px !important;
            box-shadow: 6px 6px 0px {COLORS['accent']} !important;
            margin-bottom: 20px !important;
        }}

        .badge-weak {{
            background-color: {COLORS['danger']} !important;
            color: white;
            padding: 4px 8px;
            border-radius: 4px;
            font-weight: bold;
            font-size: 12px;
            border: 1px solid white;
        }}
        .badge-average {{
            background-color: #f59e0b !important;
            color: white;
            padding: 4px 8px;
            border-radius: 4px;
            font-weight: bold;
            font-size: 12px;
            border: 1px solid white;
        }}
        .badge-good {{
            background-color: {COLORS['success']} !important;
            color: white;
            padding: 4px 8px;
            border-radius: 4px;
            font-weight: bold;
            font-size: 12px;
            border: 1px solid white;
        }}
    </style>
    """, unsafe_allow_html=True)
''')

# 4. Write auth.py
with open("auth.py", "w", encoding="utf-8") as f:
    f.write('''import os
import time
import datetime
import smtplib
import secrets
import jwt
import bcrypt
import streamlit as st
from email.utils import formatdate, make_msgid
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart

import db
import ui_theme as ut

JWT_SECRET = os.getenv("JWT_SECRET", "super-secret-infosys-key-2026")
SENDER_EMAIL = os.getenv("EMAIL_ADDRESS", "springboardportal@gmail.com")
EMAIL_PASSWORD = os.getenv("EMAIL_PASSWORD", "")
OTP_EXPIRY_MINUTES = 5

def validate_email_format(email):
    if '@' not in email: return False
    parts = email.split('@')
    if len(parts) != 2: return False
    local, domain = parts[0], parts[1]
    local_letters = sum(1 for c in local if c.isalpha())
    if local_letters < 2: return False
    if '.' not in domain: return False
    dot_idx = domain.rfind('.')
    domain_p1, domain_p2 = domain[:dot_idx], domain[dot_idx+1:]
    return sum(1 for c in domain_p1 if c.isalpha()) >= 2 and sum(1 for c in domain_p2 if c.isalpha()) >= 2

def check_password_strength(password):
    length = len(password)
    if length == 0: return "None", False, "gray", ""
    elif length < 5: return "Weak", False, "red", "Weak (Registration Blocked)"
    elif length < 10: return "Average", True, "orange", "Average (Registration Allowed)"
    else: return "Good", True, "green", "Good (Registration Allowed)"

def render_password_strength_ui(password):
    rating, is_allowed, color, label = check_password_strength(password)
    if rating == "None": return
    if rating == "Weak": st.markdown(f'<span class="badge-weak">{label}</span>', unsafe_allow_html=True)
    elif rating == "Average": st.markdown(f'<span class="badge-average">{label}</span>', unsafe_allow_html=True)
    elif rating == "Good": st.markdown(f'<span class="badge-good">{label}</span>', unsafe_allow_html=True)

def make_jwt(email, role="Viewer"):
    payload = {
        "email": email, "role": role,
        "exp": datetime.datetime.utcnow() + datetime.timedelta(hours=2), "iat": datetime.datetime.utcnow()
    }
    return jwt.encode(payload, JWT_SECRET, algorithm="HS256")

def verify_jwt(token):
    try: return jwt.decode(token, JWT_SECRET, algorithms=["HS256"])
    except: return None

def generate_otp():
    return f"{secrets.randbelow(900000) + 100000}"

def make_otp_token(email, otp):
    otp_hash = bcrypt.hashpw(otp.encode('utf-8'), bcrypt.gensalt()).decode('utf-8')
    payload = {
        "sub": email, "otp_hash": otp_hash, "type": "password_reset_otp",
        "iat": datetime.datetime.utcnow(), "exp": datetime.datetime.utcnow() + datetime.timedelta(minutes=OTP_EXPIRY_MINUTES)
    }
    return jwt.encode(payload, JWT_SECRET, algorithm="HS256")

def verify_otp_token(token, input_otp, email):
    try:
        payload = jwt.decode(token, JWT_SECRET, algorithms=["HS256"])
        if payload.get("sub") != email or payload.get("type") != "password_reset_otp":
            return False, "Security token mismatch."
        stored_hash = payload.get("otp_hash")
        if bcrypt.checkpw(input_otp.encode('utf-8'), stored_hash.encode('utf-8')):
            return True, "Valid"
        return False, "Invalid 6-digit OTP code."
    except jwt.ExpiredSignatureError:
        return False, f"This OTP code expired after {OTP_EXPIRY_MINUTES} minutes. Please request a new one."
    except:
        return False, "Invalid or expired verification token."

def send_otp_email(to_email, otp):
    if not EMAIL_PASSWORD or not SENDER_EMAIL:
        return False, "SMTP email credentials not set. Add EMAIL_ADDRESS and EMAIL_PASSWORD to Secrets."
    msg = MIMEMultipart('alternative')
    msg['From'] = f"Infosys Logistics Support <{SENDER_EMAIL}>"
    msg['To'] = to_email
    msg['Subject'] = "Infosys Freight Portal - OTP Verification Code"
    msg['Date'] = formatdate(localtime=True)
    msg['Message-ID'] = make_msgid()
    msg['Reply-To'] = SENDER_EMAIL
    text_body = f"Your verification code is: {otp}\\nExpires in {OTP_EXPIRY_MINUTES} minutes.\\n"
    html_body = f"""
    <html>
    <body style="font-family: Arial, sans-serif; text-align: center; background-color: #09090b; color: #ffffff; padding: 20px;">
        <div style="max-width: 400px; margin: 0 auto; background-color: #18181b; border: 2px solid #ffffff; border-radius: 12px; padding: 30px; box-shadow: 5px 5px 0px #a855f7;">
            <h2 style="color: #ffffff; margin-bottom: 5px;">⚡ Verification Code</h2>
            <p style="color: #a1a1aa;">Your password reset code is:</p>
            <h1 style="background: #a855f7; color: #ffffff; padding: 15px 25px; display: inline-block; border: 2px solid #ffffff; border-radius: 8px; box-shadow: 4px 4px 0px #06b6d4; letter-spacing: 4px;">{otp}</h1>
            <p style="color: #a1a1aa; font-size: 13px;">Valid for <b>{OTP_EXPIRY_MINUTES} minutes</b>.</p>
        </div>
    </body>
    </html>
    """
    msg.attach(MIMEText(text_body, 'plain'))
    msg.attach(MIMEText(html_body, 'html'))
    try:
        server = smtplib.SMTP('smtp.gmail.com', 587)
        server.starttls()
        server.login(SENDER_EMAIL, EMAIL_PASSWORD)
        server.sendmail(SENDER_EMAIL, to_email, msg.as_string())
        server.quit()
        return True, "Email sent."
    except Exception as e:
        return False, str(e)

def render_auth_header(title, subtitle="Secure Authentication Portal"):
    st.markdown(f"""
    <div style="text-align:center; padding:1rem 0;">
        <div style="font-size:48px; margin-bottom:10px;">⚡</div>
        <h1 style="font-size:2.4rem !important; margin:0; letter-spacing:-1px;">Infosys Freight Portal</h1>
        <p style="color:{ut.COLORS['text_muted']}; font-size:14px; margin-top:5px;">{subtitle}</p>
    </div>
    <div style="text-align:center; margin-bottom:1.5rem;">
        <span style="font-size:1.1rem; font-weight:800; color:{ut.COLORS['text_heading']}; border-bottom: 3px solid {ut.COLORS['accent']}; padding-bottom: 2px;">{title}</span>
    </div>
    """, unsafe_allow_html=True)

def show_login_page():
    render_auth_header("Sign in to your account", "Welcome back! Enter credentials below")
    email = st.text_input("Username or Email address", placeholder="you@infosys.com", key="login_email").lower().strip()
    password = st.text_input("Password", type="password", placeholder="••••••••", key="login_pwd")

    st.markdown("<br>", unsafe_allow_html=True)
    if st.button("Sign In →", use_container_width=True, key="btn_signin"):
        if not email or not password:
            st.error("⚠️ All fields are mandatory.")
        else:
            is_locked, status_type, rem_time = db.check_lockout_status(email)
            if is_locked:
                if status_type == 'perm_locked':
                    st.error("❌ Account is permanently locked. Please contact the administrator.")
                else:
                    st.error(f"❌ Account temporarily locked. Try again in {rem_time} seconds.")
            else:
                admin_email = os.getenv("ADMIN_EMAIL", "admin@portal.com")
                admin_pwd = os.getenv("ADMIN_PASSWORD", "AdminPassword@2026")

                if email == admin_email and password == admin_pwd:
                    db.reset_login_attempts(email)
                    st.session_state.token = make_jwt(email, role="Admin")
                    st.success("✅ Admin login successful!")
                    time.sleep(1)
                    st.rerun()
                elif db.authenticate_user(email, password):
                    user_info = db.get_user_by_email(email)
                    st.session_state.token = make_jwt(email, role=user_info[1])
                    st.success("✅ Signed in successfully!")
                    time.sleep(1)
                    st.rerun()
                else:
                    is_l, st_t, r_t = db.handle_failed_login(email)
                    if is_l:
                        if st_t == 'perm_locked':
                            st.error("❌ Invalid credentials. Account is now PERMANENTLY locked.")
                        else:
                            st.error(f"❌ Invalid credentials. Account locked for {r_t} seconds.")
                    else:
                        st.error("❌ Invalid credentials.")

    st.markdown("<hr style='border:1px dashed #3f3f46;'>", unsafe_allow_html=True)
    col1, col2 = st.columns(2)
    if col1.button("Create Account", use_container_width=True, key="btn_to_signup"):
        st.session_state.page = "Signup"
        st.rerun()
    if col2.button("Forgot Password?", use_container_width=True, key="btn_to_forgot"):
        st.session_state.forgot_step = 1
        st.session_state.page = "Forgot"
        st.rerun()

def show_signup_page():
    render_auth_header("Create your Account", "Join the Infosys Freight Systems Portal")
    username = st.text_input("Unique Username", placeholder="e.g. JaneDoe", key="signup_username")
    email = st.text_input("Email address", placeholder="you@infosys.com", key="signup_email").lower().strip()
    password = st.text_input("Password", type="password", placeholder="Min. 5 characters", key="signup_pwd")
    render_password_strength_ui(password)
    confirm_pwd = st.text_input("Confirm Password", type="password", placeholder="Re-enter password", key="signup_cpwd")
    sq = st.selectbox("Security Question", ["What is your pet name?", "What is your mother's maiden name?", "What is your favourite city?"], key="signup_sq")
    sa = st.text_input("Your Security Answer", key="signup_sa")

    st.markdown("<br>", unsafe_allow_html=True)
    if st.button("Register & Login →", use_container_width=True, key="btn_signup"):
        if not username or not email or not password or not confirm_pwd or not sa:
            st.error("⚠️ All fields are mandatory.")
        elif not validate_email_format(email):
            st.error("❌ Email format invalid.")
        elif password != confirm_pwd:
            st.error("❌ Passwords do not match.")
        else:
            rating, is_allowed, _, _ = check_password_strength(password)
            if not is_allowed:
                st.error("❌ Password strength is Weak. Please use at least 5 characters.")
            elif db.check_user_exists(email):
                st.error("❌ Email already registered.")
            elif db.check_username_exists(username):
                st.error("❌ Username already taken.")
            else:
                role = "Logistics Manager"
                if db.register_user(email, username, password, sq, sa, role):
                    st.session_state.token = make_jwt(email, role=role)
                    st.success("🎉 Account created!")
                    time.sleep(1)
                    st.session_state.page = "Home"
                    st.rerun()
                else:
                    st.error("❌ Database insertion error.")

    st.markdown("<br>", unsafe_allow_html=True)
    if st.button("← Back to Sign In", use_container_width=True, key="btn_back_login"):
        st.session_state.page = "Login"
        st.rerun()

def show_forgot_page():
    render_auth_header("Reset Password", "Recover access to your account")
    if st.session_state.forgot_step == 1:
        email = st.text_input("Enter Registered Email", placeholder="you@infosys.com", key="forgot_email").lower().strip()
        st.markdown("<br>", unsafe_allow_html=True)
        col_sq, col_otp = st.columns(2)

        if col_sq.button("Use Security Question", use_container_width=True, key="btn_forgot_sq"):
            if not email: st.error("⚠️ Please enter your email.")
            elif not db.check_user_exists(email): st.error("❌ Email not registered.")
            else:
                user_info = db.get_user_by_email(email)
                if user_info[4] == 'perm_locked':
                    st.error("❌ This account is permanently locked. Contact an admin to unlock.")
                else:
                    st.session_state.reset_email = email
                    st.session_state.reset_mode = "sq"
                    st.session_state.forgot_step = 2
                    st.session_state.sq_question = user_info[2]
                    st.rerun()

        if col_otp.button("Send Email OTP", use_container_width=True, key="btn_forgot_otp"):
            if not email: st.error("⚠️ Please enter your email.")
            elif not db.check_user_exists(email): st.error("❌ Email not registered.")
            else:
                user_info = db.get_user_by_email(email)
                if user_info[4] == 'perm_locked':
                    st.error("❌ This account is permanently locked. Contact an admin to unlock.")
                else:
                    allowed, rem_time, count = db.check_otp_resend_cooldown(email)
                    if not allowed:
                        st.warning(f"⚠️ OTP rate limit exceeded. Please wait {rem_time} seconds before resending.")
                    else:
                        otp_code = generate_otp()
                        with st.spinner("Sending 6-digit OTP..."):
                            sent, msg = send_otp_email(email, otp_code)
                        if sent:
                            db.record_otp_sent(email)
                            st.session_state.reset_email = email
                            st.session_state.reset_mode = "otp"
                            st.session_state.temp_otp_token = make_otp_token(email, otp_code)
                            st.session_state.forgot_step = 2
                            st.success("✅ OTP Sent! Check your inbox.")
                            time.sleep(1)
                            st.rerun()
                        else:
                            st.error(f"❌ SMTP Error: {msg}")

    elif st.session_state.forgot_step == 2:
        st.write(f"Recovering Account: **{st.session_state.reset_email}**")
        is_identity_verified = False

        if st.session_state.reset_mode == "sq":
            st.info(f"❓ Security Question: {st.session_state.sq_question}")
            user_answer = st.text_input("Your Security Answer", key="forgot_sa").lower().strip()
        else:
            st.info("📧 Check your email inbox for the 6-digit code.")
            otp_input = st.text_input("6-Digit Verification Code", max_chars=6, key="forgot_otp_code")

            allowed, rem_time, count = db.check_otp_resend_cooldown(st.session_state.reset_email)
            resend_label = f"Resend OTP Code" if allowed else f"Resend OTP ({rem_time}s)"
            if st.button(resend_label, disabled=not allowed, key="forgot_otp_resend"):
                otp_code = generate_otp()
                with st.spinner("Sending new 6-digit OTP..."):
                    sent, msg = send_otp_email(st.session_state.reset_email, otp_code)
                if sent:
                    db.record_otp_sent(st.session_state.reset_email)
                    st.session_state.temp_otp_token = make_otp_token(st.session_state.reset_email, otp_code)
                    st.success("✅ A new OTP code was sent!")
                    time.sleep(1)
                    st.rerun()
                else:
                    st.error(f"❌ SMTP Error: {msg}")

        new_pwd = st.text_input("New Password", type="password", key="forgot_npwd")
        render_password_strength_ui(new_pwd)
        confirm_npwd = st.text_input("Confirm New Password", type="password", key="forgot_cnpwd")

        st.markdown("<br>", unsafe_allow_html=True)
        if st.button("Reset Password →", use_container_width=True, key="btn_forgot_reset"):
            if not new_pwd or not confirm_npwd: st.error("⚠️ Please fill in passwords.")
            elif new_pwd != confirm_npwd: st.error("❌ Passwords do not match.")
            else:
                rating, is_allowed, _, _ = check_password_strength(new_pwd)
                if not is_allowed:
                    st.error("❌ Password is too weak. Must be at least 5 characters.")
                elif db.check_password_reused(st.session_state.reset_email, new_pwd):
                    st.error("❌ You cannot reuse a previous password hash.")
                else:
                    if st.session_state.reset_mode == "sq":
                        user_info = db.get_user_by_email(st.session_state.reset_email)
                        if db.check_hash(user_answer, user_info[3]): is_identity_verified = True
                        else: st.error("❌ Incorrect security answer.")
                    else:
                        verified, otp_msg = verify_otp_token(st.session_state.temp_otp_token, otp_input, st.session_state.reset_email)
                        if verified: is_identity_verified = True
                        else: st.error(f"❌ {otp_msg}")

                    if is_identity_verified:
                        db.update_password(st.session_state.reset_email, new_pwd)
                        db.reset_login_attempts(st.session_state.reset_email)
                        db.reset_otp_cooldown(st.session_state.reset_email)
                        st.success("🎉 Password updated!")
                        time.sleep(1)
                        st.session_state.reset_email = None
                        st.session_state.reset_mode = None
                        st.session_state.temp_otp_token = None
                        st.session_state.forgot_step = 1
                        st.session_state.page = "Login"
                        st.rerun()

    st.markdown("<br>", unsafe_allow_html=True)
    if st.button("← Cancel & Back to Login", use_container_width=True, key="btn_forgot_cancel"):
        st.session_state.reset_email = None
        st.session_state.reset_mode = None
        st.session_state.temp_otp_token = None
        st.session_state.forgot_step = 1
        st.session_state.page = "Login"
        st.rerun()
''')

# 5. Write admin_dash.py
with open("admin_dash.py", "w", encoding="utf-8") as f:
    f.write('''import streamlit as st
import db
import ui_theme as ut
import time

def show_admin_dashboard():
    st.markdown(f"""
    <div style="background:{ut.COLORS['bg_card']}; border: 3px solid {ut.COLORS['border']}; border-radius:16px; padding:24px; margin-bottom:24px; box-shadow: 5px 5px 0px {ut.COLORS['accent']};">
        <h1 style="color:{ut.COLORS['accent']} !important; margin:0; font-size:24px !important;">🛡️ ADMINISTRATIVE COMMAND CONTROL</h1>
        <div style="color:{ut.COLORS['text_muted']}; font-size:13px; font-weight:500;">Role Configuration, Account Lockouts, and System Audit Logs</div>
    </div>
    """, unsafe_allow_html=True)

    users = db.get_all_users()
    total_users = len(users)
    locked_users = sum(1 for u in users if u[4] == 'perm_locked')
    active_users = total_users - locked_users

    col1, col2, col3 = st.columns(3)
    with col1: st.markdown(f'<div class="nb-card" style="text-align:center;">👥 <h2>{total_users}</h2>Total Accounts</div>', unsafe_allow_html=True)
    with col2: st.markdown(f'<div class="nb-card" style="text-align:center;">🔓 <h2>{active_users}</h2>Active Roles</div>', unsafe_allow_html=True)
    with col3: st.markdown(f'<div class="nb-card" style="text-align:center;">🔒 <h2>{locked_users}</h2>Locked Accounts</div>', unsafe_allow_html=True)

    tab_users, tab_add, tab_roles = st.tabs(["📋 USER MANAGEMENT", "➕ REGISTER NEW USER", "🛡️ ASSIGN ROLES"])

    with tab_users:
        st.markdown("### Active Credentials & Security Status")
        search = st.text_input("Filter by username...", "", key="admin_user_search").lower().strip()
        filtered_users = [u for u in users if search in u[0].lower() or search in u[1].lower()]

        if not filtered_users:
            st.info("No registered users match your search criteria.")
        else:
            for idx, u in enumerate(filtered_users):
                username, email, role, failed_attempts, status, created_at = u
                with st.container(border=True):
                    col_info, col_status, col_actions = st.columns([4, 2, 2.5])
                    with col_info:
                        st.markdown(f"**Username:** {username} | `{email}`")
                        st.markdown(f"**Registered:** {created_at}")
                    with col_status:
                        st.markdown(f"**Role:** `{role}`")
                        if status == 'perm_locked': st.markdown("🔴 **PERM LOCKED**")
                        else: st.markdown(f"🟢 **ACTIVE** (Fails: {failed_attempts})")
                    with col_actions:
                        if status in ['perm_locked', 'temp_locked'] or failed_attempts > 0:
                            if st.button("Unlock / Reset", key=f"btn_unlock_{idx}", use_container_width=True):
                                db.unlock_user(email)
                                st.toast(f"Unlocked account: {username}")
                                time.sleep(1)
                                st.rerun()
                        current_admin = st.session_state.get("token_payload", {}).get("email", "")
                        if email.lower() != current_admin.lower() and email.lower() != "admin@portal.com":
                            if st.button("Delete User", key=f"btn_del_{idx}", use_container_width=True):
                                db.delete_user(email)
                                st.toast(f"Deleted: {username}")
                                time.sleep(1)
                                st.rerun()

    with tab_add:
        st.markdown("### Create Account Database Entry")
        with st.form("new_user_form", clear_on_submit=True):
            nu_username = st.text_input("Username", placeholder="e.g. logistix_pro")
            nu_email = st.text_input("Email Address", placeholder="e.g. user@logisticsco.com").lower().strip()
            nu_pwd = st.text_input("Password", type="password", placeholder="••••••••")
            nu_role = st.selectbox("Assign Role", ["Viewer", "Logistics Manager", "Admin"])
            nu_sq = st.selectbox("Security Question", ["What is your pet name?", "What is your mother's maiden name?", "What is your favourite city?"])
            nu_sa = st.text_input("Security Answer")
            submit_nu = st.form_submit_button("Register Account System Node")
            if submit_nu:
                if not nu_username or not nu_email or not nu_pwd or not nu_sa:
                    st.error("⚠️ All fields are mandatory.")
                elif len(nu_pwd) < 5:
                    st.error("❌ Password must be at least 5 characters.")
                elif db.check_user_exists(nu_email):
                    st.error("❌ Email already registered.")
                else:
                    if db.register_user(nu_email, nu_username, nu_pwd, nu_sq, nu_sa, nu_role):
                        st.success(f"🎉 Created user {nu_username}!")
                        time.sleep(1)
                        st.rerun()

    with tab_roles:
        st.markdown("### Modify Security Permissions")
        role_email = st.selectbox("Select Target Email Node", [u[1] for u in users], key="role_email_sel")
        target_user = next((u for u in users if u[1] == role_email), None)
        if target_user:
            st.markdown(f"**Current Role:** `{target_user[2]}`")
            new_role = st.selectbox("Select New Target Role", ["Viewer", "Logistics Manager", "Admin"], index=["Viewer", "Logistics Manager", "Admin"].index(target_user[2]))
            if st.button("Update Security Role", use_container_width=True):
                current_admin = st.session_state.get("token_payload", {}).get("email", "")
                if role_email.lower() == current_admin.lower() and new_role != "Admin":
                    st.error("❌ You cannot downgrade your own administrator security role.")
                else:
                    db.update_user_role(role_email, new_role)
                    st.toast(f"Role updated to {new_role}")
                    time.sleep(1)
                    st.rerun()
''')

# 6. Write train_ml_freight.py
with open("train_ml_freight.py", "w", encoding="utf-8") as f:
    f.write('''import os
import joblib
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_squared_error, roc_auc_score, accuracy_score
import db

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor, AdaBoostRegressor
from sklearn.linear_model import Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier, AdaBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

MODELS_DIR = os.path.join(os.path.dirname(os.path.abspath(__file__)), "models")
os.makedirs(MODELS_DIR, exist_ok=True)

INDIAN_PORTS = ["JNPT Mumbai", "Mundra", "Chennai", "Cochin"]
GLOBAL_PORTS = ["Shanghai", "Rotterdam", "Singapore", "New York"]

def generate_synthetic_pricing_data(n_samples=1000):
    np.random.seed(42)
    origins = np.random.choice(GLOBAL_PORTS, size=n_samples)
    destinations = np.random.choice(INDIAN_PORTS, size=n_samples)

    distance_map = {
        ("Shanghai", "JNPT Mumbai"): 5000, ("Shanghai", "Mundra"): 5200,
        ("Shanghai", "Chennai"): 4200, ("Shanghai", "Cochin"): 4500,
        ("Rotterdam", "JNPT Mumbai"): 7200, ("Rotterdam", "Mundra"): 7000,
        ("Rotterdam", "Chennai"): 8200, ("Rotterdam", "Cochin"): 7900,
        ("Singapore", "JNPT Mumbai"): 3000, ("Singapore", "Mundra"): 3300,
        ("Singapore", "Chennai"): 1800, ("Singapore", "Cochin"): 2200,
        ("New York", "JNPT Mumbai"): 11500, ("New York", "Mundra"): 11200,
        ("New York", "Chennai"): 12000, ("New York", "Cochin"): 11800,
    }

    distances = [distance_map.get((o, d), 5000) + np.random.uniform(-100, 100) for o, d in zip(origins, destinations)]
    distances = np.array(distances)

    cargo_weight = np.random.uniform(1.0, 30.0, size=n_samples)
    port_congestion = np.random.uniform(0.1, 1.0, size=n_samples)
    fuel_index = np.random.uniform(0.7, 1.5, size=n_samples)
    carrier_rating = np.random.uniform(1.0, 5.0, size=n_samples)

    noise = np.random.normal(0, 25, size=n_samples)
    prices = 800 + 0.65 * distances + 120.0 * cargo_weight + 850.0 * port_congestion + 1100.0 * fuel_index - 50.0 * carrier_rating + noise

    return pd.DataFrame({
        "origin_port": origins, "destination_port": destinations, "distance_nm": distances,
        "cargo_weight_tons": cargo_weight, "port_congestion_index": port_congestion,
        "fuel_price_index": fuel_index, "carrier_rating": carrier_rating, "freight_cost_usd": prices
    })

def generate_synthetic_delay_data(n_samples=1000):
    np.random.seed(43)
    origins = np.random.choice(GLOBAL_PORTS, size=n_samples)
    destinations = np.random.choice(INDIAN_PORTS, size=n_samples)
    distance = np.random.uniform(1500, 12000, size=n_samples)
    weather_risk = np.random.uniform(0.0, 1.0, size=n_samples)
    port_congestion = np.random.uniform(0.0, 1.0, size=n_samples)
    carrier_reliability = np.random.uniform(0.5, 1.0, size=n_samples)
    ship_age = np.random.uniform(1.0, 25.0, size=n_samples)

    logit = -1.5 + 0.00015 * distance + 3.5 * weather_risk + 2.5 * port_congestion - 4.0 * carrier_reliability + 0.05 * ship_age
    prob = 1 / (1 + np.exp(-logit))
    delays = (prob > 0.5).astype(int)

    return pd.DataFrame({
        "origin_port": origins, "destination_port": destinations, "distance_nm": distance,
        "weather_risk_index": weather_risk, "port_congestion_index": port_congestion,
        "carrier_reliability": carrier_reliability, "ship_age_years": ship_age, "is_delayed": delays
    })

def generate_synthetic_compliance_data(n_samples=1000):
    np.random.seed(44)
    carriers = [f"Carrier_{i:02d}" for i in range(1, 21)]
    carrier_names = np.random.choice(carriers, size=n_samples)

    on_time_rate = np.random.uniform(0.5, 1.0, size=n_samples)
    safety_violations = np.random.choice([0, 1, 2, 3, 4, 5], size=n_samples, p=[0.6, 0.2, 0.1, 0.05, 0.03, 0.02])
    maintenance_status = np.random.choice([1, 0], size=n_samples, p=[0.85, 0.15])
    feedback_rating = np.random.uniform(1.0, 5.0, size=n_samples)
    insurance_valid = np.random.choice([1, 0], size=n_samples, p=[0.95, 0.05])

    logit = 1.5 + 5.0 * on_time_rate - 2.5 * safety_violations + 3.0 * maintenance_status + 0.8 * feedback_rating + 4.0 * insurance_valid - 8.0
    prob = 1 / (1 + np.exp(-logit))
    compliance = (prob > 0.5).astype(int)

    return pd.DataFrame({
        "carrier_name": carrier_names, "on_time_delivery_rate": on_time_rate,
        "safety_violations_count": safety_violations, "maintenance_check_passed": maintenance_status,
        "feedback_rating": feedback_rating, "insurance_valid": insurance_valid, "is_compliant": compliance
    })

def train_agent1_pricing():
    print("\\n--- Training Agent 1: Dynamic Pricing (Regression) ---")
    df = generate_synthetic_pricing_data()
    le_orig = LabelEncoder()
    le_dest = LabelEncoder()
    df["origin_encoded"] = le_orig.fit_transform(df["origin_port"])
    df["dest_encoded"] = le_dest.fit_transform(df["destination_port"])

    joblib.dump(le_orig, os.path.join(MODELS_DIR, "encoder_origin.joblib"))
    joblib.dump(le_dest, os.path.join(MODELS_DIR, "encoder_dest.joblib"))

    X = df[["distance_nm", "cargo_weight_tons", "port_congestion_index", "fuel_price_index", "carrier_rating", "origin_encoded", "dest_encoded"]]
    y = df["freight_cost_usd"]

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    joblib.dump(scaler, os.path.join(MODELS_DIR, "scaler_pricing.joblib"))

    regressors = {
        "RandomForestRegressor": RandomForestRegressor(n_estimators=100, random_state=42),
        "GradientBoostingRegressor": GradientBoostingRegressor(random_state=42),
        "ExtraTreesRegressor": ExtraTreesRegressor(n_estimators=100, random_state=42),
        "RidgeRegression": Ridge(),
        "DecisionTreeRegressor": DecisionTreeRegressor(random_state=42),
        "AdaBoostRegressor": AdaBoostRegressor(random_state=42),
        "KNeighborsRegressor": KNeighborsRegressor()
    }

    best_algo, best_r2, best_rmse, best_model = None, -1.0, float('inf'), None
    for name, model in regressors.items():
        model.fit(X_train_scaled, y_train)
        preds = model.predict(X_test_scaled)
        r2 = r2_score(y_test, preds)
        rmse = np.sqrt(mean_squared_error(y_test, preds))
        print(f"-> {name} | R2: {r2:.4f} | RMSE: {rmse:.4f}")
        if r2 > best_r2:
            best_r2, best_rmse, best_algo, best_model = r2, rmse, name, model

    print(f"[Best Pricing Model] {best_algo} with R2: {best_r2:.4f}")
    model_path = os.path.join(MODELS_DIR, "best_pricing_model.joblib")
    joblib.dump(best_model, model_path)
    db.log_trained_model("Agent 1: Dynamic Pricing", best_algo, None, best_r2, best_rmse, None, model_path)
    return best_algo, best_r2, best_rmse

def train_agent2_delay():
    print("\\n--- Training Agent 2: Route Delay Prediction (Classification) ---")
    df = generate_synthetic_delay_data()
    le_orig = LabelEncoder()
    le_dest = LabelEncoder()
    df["origin_encoded"] = le_orig.fit_transform(df["origin_port"])
    df["dest_encoded"] = le_dest.fit_transform(df["destination_port"])

    joblib.dump(le_orig, os.path.join(MODELS_DIR, "encoder_delay_origin.joblib"))
    joblib.dump(le_dest, os.path.join(MODELS_DIR, "encoder_delay_dest.joblib"))

    X = df[["distance_nm", "weather_risk_index", "port_congestion_index", "carrier_reliability", "ship_age_years", "origin_encoded", "dest_encoded"]]
    y = df["is_delayed"]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    joblib.dump(scaler, os.path.join(MODELS_DIR, "scaler_delay.joblib"))

    classifiers = {
        "RandomForestClassifier": RandomForestClassifier(n_estimators=100, random_state=42),
        "GradientBoostingClassifier": GradientBoostingClassifier(random_state=42),
        "LogisticRegression": LogisticRegression(random_state=42),
        "SVC": SVC(probability=True, random_state=42),
        "ExtraTreesClassifier": ExtraTreesClassifier(n_estimators=100, random_state=42),
        "AdaBoostClassifier": AdaBoostClassifier(random_state=42),
        "KNeighborsClassifier": KNeighborsClassifier()
    }

    best_algo, best_auc, best_acc, best_model = None, -1.0, 0.0, None
    for name, model in classifiers.items():
        model.fit(X_train_scaled, y_train)
        preds = model.predict(X_test_scaled)
        probs = model.predict_proba(X_test_scaled)[:, 1]
        auc = roc_auc_score(y_test, probs)
        acc = accuracy_score(y_test, preds)
        print(f"-> {name} | ROC-AUC: {auc:.4f} | Accuracy: {acc:.4f}")
        if auc > best_auc:
            best_auc, best_acc, best_algo, best_model = auc, acc, name, model

    print(f"[Best Delay Model] {best_algo} with ROC-AUC: {best_auc:.4f}")
    model_path = os.path.join(MODELS_DIR, "best_delay_model.joblib")
    joblib.dump(best_model, model_path)
    db.log_trained_model("Agent 2: Route Delay Prediction", best_algo, best_acc, None, None, best_auc, model_path)
    return best_algo, best_auc, best_acc

def train_agent3_compliance():
    print("\\n--- Training Agent 3: Carrier Compliance (Classification) ---")
    df = generate_synthetic_compliance_data()
    le_carrier = LabelEncoder()
    df["carrier_encoded"] = le_carrier.fit_transform(df["carrier_name"])
    joblib.dump(le_carrier, os.path.join(MODELS_DIR, "encoder_compliance_carrier.joblib"))

    X = df[["on_time_delivery_rate", "safety_violations_count", "maintenance_check_passed", "feedback_rating", "insurance_valid", "carrier_encoded"]]
    y = df["is_compliant"]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    joblib.dump(scaler, os.path.join(MODELS_DIR, "scaler_compliance.joblib"))

    classifiers = {
        "GradientBoostingClassifier": GradientBoostingClassifier(random_state=42),
        "RandomForestClassifier": RandomForestClassifier(n_estimators=100, random_state=42),
        "ExtraTreesClassifier": ExtraTreesClassifier(n_estimators=100, random_state=42),
        "LogisticRegression": LogisticRegression(random_state=42),
        "AdaBoostClassifier": AdaBoostClassifier(random_state=42),
        "SVC": SVC(probability=True, random_state=42)
    }

    best_algo, best_auc, best_acc, best_model = None, -1.0, 0.0, None
    for name, model in classifiers.items():
        model.fit(X_train_scaled, y_train)
        preds = model.predict(X_test_scaled)
        probs = model.predict_proba(X_test_scaled)[:, 1]
        auc = roc_auc_score(y_test, probs)
        acc = accuracy_score(y_test, preds)
        print(f"-> {name} | ROC-AUC: {auc:.4f} | Accuracy: {acc:.4f}")
        if auc > best_auc:
            best_auc, best_acc, best_algo, best_model = auc, acc, name, model

    print(f"[Best Compliance Model] {best_algo} with ROC-AUC: {best_auc:.4f}")
    model_path = os.path.join(MODELS_DIR, "best_compliance_model.joblib")
    joblib.dump(best_model, model_path)
    db.log_trained_model("Agent 3: Carrier Compliance", best_algo, best_acc, None, None, best_auc, model_path)
    return best_algo, best_auc, best_acc

def train_all_agents():
    db.init_db()
    pricing_algo, pricing_r2, pricing_rmse = train_agent1_pricing()
    delay_algo, delay_auc, delay_acc = train_agent2_delay()
    compliance_algo, compliance_auc, compliance_acc = train_agent3_compliance()
    print("\\n==============================================")
    print("All Machine Learning Agents Successfully Trained!")
    print("==============================================")
    print(f"Agent 1: Pricing Best = {pricing_algo} | R2: {pricing_r2:.4f}")
    print(f"Agent 2: Delay Best   = {delay_algo} | AUC: {delay_auc:.4f}")
    print(f"Agent 3: Compl. Best  = {compliance_algo} | AUC: {compliance_acc:.4f}")
    print("==============================================\\n")

if __name__ == "__main__":
    train_all_agents()
''')

# 7. Write llm_engine_freight.py
with open("llm_engine_freight.py", "w", encoding="utf-8") as f:
    f.write('''import os
import json
import time

GPU_AVAILABLE = False
try:
    import torch
    if torch.cuda.is_available(): GPU_AVAILABLE = True
except ImportError:
    pass

HF_TOKEN = os.getenv("HF_TOKEN", "")
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

model = None
tokenizer = None
llm_loaded = False

def initialize_llm():
    global model, tokenizer, llm_loaded
    if not GPU_AVAILABLE:
        print("GPU unavailable. Rule-based Copilot Fallback activated.")
        return False
    try:
        from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16
        )
        tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN if HF_TOKEN else None)
        model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb_config, device_map="auto", token=HF_TOKEN if HF_TOKEN else None)
        llm_loaded = True
        return True
    except Exception as e:
        print(f"Error loading HF: {e}. Falling back.")
        return False

initialize_llm()

LOGISTICS_KNOWLEDGE_BASE = {
    "port congestion": "Port congestion increases freight risk by causing vessel delays, demurrage charges, and yard bottlenecks. In ports like JNPT Mumbai and Mundra, congestion stops quick container turnarounds, raising pricing spikes.",
    "jnpt": "JNPT Mumbai is India's largest container port, handling high volume with automation, but faces seasonal monsoon delays and local highway gridlocks.",
    "mundra": "Mundra Port in Gujarat is a deep-water private hub. It offers quick turnaround times but occasionally faces Northern rail bottleneck challenges.",
    "chennai": "Chennai Port is the southeastern gateway, handling automotive and general containers, but truck access constraints limit port gates during standard day hours.",
    "cochin": "Cochin Port hosts the Vallarpadam Terminal. It is close to Suez shipping lanes but faces feeder connectivity and labor union dynamics.",
    "default": "Maritime quote calculations depend on vessel capacity, routing congestion, and fuel adjustment indexes. Using ML models helps minimize risks on ports like Mundra and JNPT Mumbai."
}

def query_copilot(prompt: str) -> str:
    global model, tokenizer, llm_loaded
    prompt_lower = prompt.lower()
    if llm_loaded and model is not None and tokenizer is not None:
        try:
            messages = [
                {"role": "system", "content": "You are a helpful logistics assistant. Provide concise answers."},
                {"role": "user", "content": prompt}
            ]
            text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            inputs = tokenizer([text], return_tensors="pt").to("cuda")
            with torch.no_grad():
                generated_ids = model.generate(inputs.input_ids, max_new_tokens=256)
            generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(inputs.input_ids, generated_ids)]
            return tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
        except:
            pass
    for key, val in LOGISTICS_KNOWLEDGE_BASE.items():
        if key in prompt_lower: return val
    return LOGISTICS_KNOWLEDGE_BASE["default"]

def generate_audit_report(pricing_usd, delay_prob, compliance_prob, origin, destination, weight_tons, carrier):
    delay_risk = "Low"
    if delay_prob > 0.6: delay_risk = "High"
    elif delay_prob > 0.3: delay_risk = "Medium"
    compliance_status = "Compliant" if compliance_prob >= 0.5 else "Non-Compliant"

    summary = f"Freight audit for shipment from {origin} to {destination} ({weight_tons} tons) carried by {carrier}. "
    summary += f"Pricing model estimates dynamic rate at ${pricing_usd:,.2f} USD. "
    if delay_risk == "High":
        summary += f"High delay risk detected ({delay_prob*100:.1f}%). Suggest checking port congestion or route alternatives. "
    else:
        summary += f"Route schedule looks stable with delay probability of {delay_prob*100:.1f}%. "
    if compliance_status == "Non-Compliant":
        summary += "WARNING: Carrier compliance check failed! Please review documentation before booking."
    else:
        summary += "Carrier compliance meets standard logistics criteria."

    return json.dumps({
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
        "shipment_details": {
            "origin": origin, "destination": destination,
            "weight_tons": float(weight_tons), "carrier": carrier
        },
        "agents_analysis": {
            "dynamic_pricing": {"estimated_freight_cost_usd": round(float(pricing_usd), 2)},
            "route_delay_predictor": {"delay_probability": round(float(delay_prob), 4), "risk_level": delay_risk},
            "carrier_compliance_inspector": {"compliance_probability": round(float(compliance_prob), 4), "status": compliance_status}
        },
        "overall_audit_summary": summary
    }, indent=2)
''')

# 8. Write app.py
with open("app.py", "w", encoding="utf-8") as f:
    f.write('''import os
import time
import streamlit as st
import joblib
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import json

import db
import ui_theme as ut
import auth
import admin_dash
import train_ml_freight as train_ml
import llm_engine_freight as llm_engine

st.set_page_config(page_title="Infosys Freight Portal", page_icon="⚡", layout="wide")
ut.inject_theme_css()

for k, v in [
    ("token", None), ("page", "Home"), ("predictions_count", 0),
    ("reset_email", None), ("reset_mode", None), ("temp_otp_token", None), ("forgot_step", 1)
]:
    if k not in st.session_state: st.session_state[k] = v

db.init_db()

MODELS_DIR = os.path.join(os.path.dirname(os.path.abspath(__file__)), "models")
pricing_model_file = os.path.join(MODELS_DIR, "best_pricing_model.joblib")
if not os.path.exists(pricing_model_file):
    with st.spinner("📦 Initializing ML models..."):
        train_ml.train_all_agents()

def navigate(to_page):
    st.session_state.page = to_page
    st.rerun()

if not st.session_state.token:
    if st.session_state.page not in ["Login", "Signup", "Forgot"]: st.session_state.page = "Login"
    _, center_col, _ = st.columns([1, 1.4, 1])
    with center_col:
        st.markdown('<div class="nb-card">', unsafe_allow_html=True)
        if st.session_state.page == "Login": auth.show_login_page()
        elif st.session_state.page == "Signup": auth.show_signup_page()
        elif st.session_state.page == "Forgot": auth.show_forgot_page()
        st.markdown('</div>', unsafe_allow_html=True)
else:
    payload = auth.verify_jwt(st.session_state.token)
    if not payload:
        st.session_state.token = None
        st.session_state.page = "Login"
        st.error("Session expired.")
        time.sleep(1)
        st.rerun()
    st.session_state.token_payload = payload
    email, role = payload["email"], payload["role"]
    username = "Administrator" if role == "Admin" else db.get_user_by_email(email)[0]

    with st.sidebar:
        st.markdown(f"""
        <div style="padding:16px 8px; text-align:center;">
            <div style="font-size:36px; margin-bottom:5px;">⚡</div>
            <div style="font-weight:800; font-size:18px; color:{ut.COLORS['text_heading']};">Infosys Freight</div>
            <div style="font-size:11px; background:{ut.COLORS['accent']}; color:{ut.COLORS['text_heading']}; font-weight:700; padding:2px 8px; border-radius:30px; display:inline-block; border: 2px solid {ut.COLORS['border']}; margin-top: 5px;">
                {role.upper()} NODE
            </div>
        </div>
        <hr style="border: 1px dashed {ut.COLORS['border']}; margin: 15px 0;">
        """, unsafe_allow_html=True)
        st.write(f"Logged in: **{username}**")

        menu = ["🏠 Home", "💵 Pricing Calculator", "⏳ Route Delay Predictor", "🛡️ Carrier Compliance", "🤖 AI Copilot", "⚙️ Settings"]
        if role == "Admin": menu.append("🛡️ Admin Dashboard")
        selected_menu = st.radio("Navigation Lanes", menu, index=menu.index("🏠 Home") if st.session_state.page == "Home" else 0)

        page_mapping = {
            "🏠 Home": "Home", "💵 Pricing Calculator": "Pricing", "⏳ Route Delay Predictor": "Delay",
            "🛡️ Carrier Compliance": "Compliance", "🤖 AI Copilot": "Copilot", "⚙️ Settings": "Settings", "🛡️ Admin Dashboard": "Admin"
        }
        if page_mapping.get(selected_menu) != st.session_state.page:
            st.session_state.page = page_mapping.get(selected_menu)
            st.rerun()
        if st.button("Logout 🚪", use_container_width=True):
            st.session_state.token = None
            st.session_state.page = "Login"
            st.rerun()

    if st.session_state.page == "Home":
        st.markdown(f"""
        <div style="background:{ut.COLORS['bg_card']}; border: 3px solid {ut.COLORS['border']}; border-radius:16px; padding:24px; margin-bottom:24px; box-shadow: 5px 5px 0px {ut.COLORS['accent']};">
            <h1 style="color:{ut.COLORS['accent']} !important; margin:0; font-size:24px !important;">⚓ LOGISTICS DASHBOARD</h1>
            <div style="color:{ut.COLORS['text_muted']}; font-size:13px;">Real-time Ocean Freight Quote Optimization System</div>
        </div>
        """, unsafe_allow_html=True)

        users_count = len(db.get_all_users())
        models_count = len(db.get_trained_models())
        predictions_today = st.session_state.predictions_count

        c1, c2, c3, c4 = st.columns(4)
        with c1: st.markdown(f'<div class="nb-card" style="text-align:center;">👤 <h3>{users_count}</h3>System Users</div>', unsafe_allow_html=True)
        with c2: st.markdown(f'<div class="nb-card" style="text-align:center;">🤖 <h3>{models_count}</h3>Models Logged</div>', unsafe_allow_html=True)
        with c3: st.markdown(f'<div class="nb-card" style="text-align:center;">🔍 <h3>{predictions_today}</h3>Queries Today</div>', unsafe_allow_html=True)
        with c4: st.markdown(f'<div class="nb-card" style="text-align:center;">🛡️ <h3>98.9%</h3>System Health</div>', unsafe_allow_html=True)

        chart_col, data_col = st.columns([1.5, 1])
        with chart_col:
            st.markdown("### Port Coverage Route Distances")
            routes = ["Mundra", "JNPT Mumbai", "Chennai", "Cochin"]
            distances = [5200, 5000, 4200, 4500]
            fig = go.Figure(go.Bar(x=routes, y=distances, marker_color=ut.COLORS['accent'], text=[f"{d} nm" for d in distances], textposition='auto'))
            fig.update_layout(title="Distance from Shanghai Hub", paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)", font=dict(color=ut.COLORS['text_main']), height=280)
            st.plotly_chart(fig, use_container_width=True)
        with data_col:
            st.markdown("### Port Coverage Information")
            st.markdown("""
            | Indian Port | Coverage Status | Core Cargo Support |
            |---|---|---|
            | **JNPT Mumbai** | 🟢 Fully Active | Containers & Bulk |
            | **Mundra** | 🟢 Fully Active | Heavy Logistics |
            | **Chennai** | 🟢 Fully Active | Auto & Machinery |
            | **Cochin** | 🟢 Fully Active | Spices & LNG |
            """)

    elif st.session_state.page == "Pricing":
        st.markdown("## Agent 1: Dynamic Pricing Calculator")
        with st.form("pricing_form"):
            col1, col2 = st.columns(2)
            with col1:
                origin = st.selectbox("Origin Port", ["Shanghai", "Rotterdam", "Singapore", "New York"])
                destination = st.selectbox("Destination Port", ["JNPT Mumbai", "Mundra", "Chennai", "Cochin"])
                weight = st.number_input("Cargo Weight (Tons)", min_value=1.0, max_value=100.0, value=15.0)
            with col2:
                congestion = st.slider("Port Congestion Index", 0.1, 1.0, 0.4)
                fuel = st.slider("Bunker Fuel Price Index", 0.5, 1.5, 1.0)
                rating = st.slider("Carrier Star Rating", 1.0, 5.0, 4.0)
            submit = st.form_submit_button("Calculate Dynamic Freight Cost")

        if submit:
            try:
                pricing_model = joblib.load(os.path.join(MODELS_DIR, "best_pricing_model.joblib"))
                scaler = joblib.load(os.path.join(MODELS_DIR, "scaler_pricing.joblib"))
                le_orig = joblib.load(os.path.join(MODELS_DIR, "encoder_origin.joblib"))
                le_dest = joblib.load(os.path.join(MODELS_DIR, "encoder_dest.joblib"))

                distance_map = {
                    ("Shanghai", "JNPT Mumbai"): 5000, ("Shanghai", "Mundra"): 5200, ("Shanghai", "Chennai"): 4200, ("Shanghai", "Cochin"): 4500,
                    ("Rotterdam", "JNPT Mumbai"): 7200, ("Rotterdam", "Mundra"): 7000, ("Rotterdam", "Chennai"): 8200, ("Rotterdam", "Cochin"): 7900,
                    ("Singapore", "JNPT Mumbai"): 3000, ("Singapore", "Mundra"): 3300, ("Singapore", "Chennai"): 1800, ("Singapore", "Cochin"): 2200,
                    ("New York", "JNPT Mumbai"): 11500, ("New York", "Mundra"): 11200, ("New York", "Chennai"): 12000, ("New York", "Cochin"): 11800,
                }
                dist = distance_map.get((origin, destination), 5000)
                orig_enc = le_orig.transform([origin])[0]
                dest_enc = le_dest.transform([destination])[0]

                features = np.array([[dist, weight, congestion, fuel, rating, orig_enc, dest_enc]])
                features_scaled = scaler.transform(features)
                pred_price = pricing_model.predict(features_scaled)[0]
                st.session_state.predictions_count += 1

                models_list = db.get_trained_models()
                pricing_stat = next((m for m in models_list if m[0] == "Agent 1: Dynamic Pricing"), (None, "Ridge", 0.0, 0.99, 29.0, 0.0))

                st.markdown("---")
                r1, r2, r3 = st.columns(3)
                with r1: st.metric(label="Predicted Ocean Freight Price", value=f"${pred_price:,.2f} USD")
                with r2: st.metric(label="Model R² Accuracy Score", value=f"{pricing_stat[3]:.4f}")
                with r3: st.metric(label="Model RMSE", value=f"${pricing_stat[4]:.2f}")
            except Exception as e: st.error(f"Error: {e}")

    elif st.session_state.page == "Delay":
        st.markdown("## Agent 2: Route Delay Predictor")
        with st.form("delay_form"):
            col1, col2 = st.columns(2)
            with col1:
                origin = st.selectbox("Origin Port", ["Shanghai", "Rotterdam", "Singapore", "New York"])
                destination = st.selectbox("Destination Port", ["JNPT Mumbai", "Mundra", "Chennai", "Cochin"])
                distance = st.number_input("Input Travel Route Distance (nm)", min_value=500, max_value=20000, value=5000)
            with col2:
                weather = st.slider("Weather Risk Index", 0.0, 1.0, 0.3)
                congestion = st.slider("Port Congestion Index", 0.0, 1.0, 0.4)
                reliability = st.slider("Historical Carrier Reliability", 0.1, 1.0, 0.85)
                ship_age = st.number_input("Carrier Vessel Age (years)", min_value=1, max_value=40, value=8)
            submit = st.form_submit_button("Predict Transit Delay Probability")

        if submit:
            try:
                delay_model = joblib.load(os.path.join(MODELS_DIR, "best_delay_model.joblib"))
                scaler = joblib.load(os.path.join(MODELS_DIR, "scaler_delay.joblib"))
                le_orig = joblib.load(os.path.join(MODELS_DIR, "encoder_delay_origin.joblib"))
                le_dest = joblib.load(os.path.join(MODELS_DIR, "encoder_delay_dest.joblib"))

                orig_enc = le_orig.transform([origin])[0]
                dest_enc = le_dest.transform([destination])[0]

                features = np.array([[distance, weather, congestion, reliability, ship_age, orig_enc, dest_enc]])
                features_scaled = scaler.transform(features)
                prob = delay_model.predict_proba(features_scaled)[0][1]
                pred_class = delay_model.predict(features_scaled)[0]
                st.session_state.predictions_count += 1

                models_list = db.get_trained_models()
                delay_stat = next((m for m in models_list if m[0] == "Agent 2: Route Delay Prediction"), (None, "LR", 0.0, 0.0, 0.0, 0.99))

                st.markdown("---")
                r1, r2, r3 = st.columns(3)
                with r1: st.metric(label="Delay Probability", value=f"{prob*100:.2f}%")
                with r2: st.metric(label="Prediction Class", value="DELAY EXPECTED" if pred_class == 1 else "ON-TIME ARRIVAL")
                with r3: st.metric(label="Classification ROC-AUC Score", value=f"{delay_stat[5]:.4f}")
            except Exception as e: st.error(f"Error: {e}")

    elif st.session_state.page == "Compliance":
        st.markdown("## Agent 3: Carrier Compliance Auditor")
        with st.form("compliance_form"):
            col1, col2 = st.columns(2)
            with col1:
                carrier = st.selectbox("Carrier Company", [f"Carrier_{i:02d}" for i in range(1, 21)])
                ontime_rate = st.slider("Historical On-Time Delivery Rate", 0.1, 1.0, 0.88)
                violations = st.number_input("Registered Safety Violations", min_value=0, max_value=10, value=0)
            with col2:
                maint_passed = st.selectbox("Maintenance Safety Status", ["Passed", "Failed"])
                feedback = st.slider("Feedback Rating (1-5)", 1.0, 5.0, 4.2)
                insurance = st.selectbox("Valid Liability Insurance", ["Yes", "No"])
            submit = st.form_submit_button("Verify Carrier Compliance")

        if submit:
            try:
                compliance_model = joblib.load(os.path.join(MODELS_DIR, "best_compliance_model.joblib"))
                scaler = joblib.load(os.path.join(MODELS_DIR, "scaler_compliance.joblib"))
                le_carrier = joblib.load(os.path.join(MODELS_DIR, "encoder_compliance_carrier.joblib"))

                carrier_enc = le_carrier.transform([carrier])[0]
                maint_val = 1 if maint_passed == "Passed" else 0
                ins_val = 1 if insurance == "Yes" else 0

                features = np.array([[ontime_rate, violations, maint_val, feedback, ins_val, carrier_enc]])
                features_scaled = scaler.transform(features)
                prob = compliance_model.predict_proba(features_scaled)[0][1]
                pred_class = compliance_model.predict(features_scaled)[0]
                st.session_state.predictions_count += 1

                models_list = db.get_trained_models()
                compliance_stat = next((m for m in models_list if m[0] == "Agent 3: Carrier Compliance"), (None, "ET", 0.0, 0.0, 0.0, 1.00))

                st.markdown("---")
                r1, r2, r3 = st.columns(3)
                with r1: st.metric(label="Compliance Probability", value=f"{prob*100:.2f}%")
                with r2: st.metric(label="Status", value="COMPLIANT" if pred_class == 1 else "NON-COMPLIANT")
                with r3: st.metric(label="ROC-AUC Score", value=f"{compliance_stat[5]:.4f}")
            except Exception as e: st.error(f"Error: {e}")

    elif st.session_state.page == "Copilot":
        st.markdown("## Logistics AI Copilot")
        tab_qa, tab_audit = st.tabs(["💬 LOGISTICS Q&A", "📋 CONSOLIDATED AUDIT REPORT"])
        with tab_qa:
            question = st.text_input("Ask a logistics query:", "Explain why port congestion increases freight risk.")
            if st.button("Query AI Copilot"):
                with st.spinner("AI thinking..."): answer = llm_engine.query_copilot(question)
                st.info(answer)
        with tab_audit:
            with st.form("copilot_audit_form"):
                col1, col2 = st.columns(2)
                with col1:
                    c_origin = st.selectbox("Origin Port", ["Shanghai", "Rotterdam", "Singapore", "New York"])
                    c_destination = st.selectbox("Destination Port", ["JNPT Mumbai", "Mundra", "Chennai", "Cochin"])
                    c_weight = st.number_input("Cargo Weight (Tons)", min_value=1.0, max_value=100.0, value=20.0)
                    c_carrier = st.selectbox("Carrier Company", [f"Carrier_{i:02d}" for i in range(1, 21)])
                with col2:
                    c_congestion = st.slider("Port Congestion Index", 0.1, 1.0, 0.45)
                    c_fuel = st.slider("Fuel Index", 0.5, 1.5, 1.1)
                    c_rating = st.slider("Carrier Star Rating", 1.0, 5.0, 4.2)
                    c_weather = st.slider("Weather Risk Index", 0.0, 1.0, 0.35)
                    c_ontime = st.slider("Carrier On-time Rate", 0.5, 1.0, 0.90)
                    c_violations = st.number_input("Safety Violations", min_value=0, max_value=5, value=0)
                    c_maint = st.selectbox("Maintenance Status", ["Passed", "Failed"])
                    c_ins = st.selectbox("Insurance Valid", ["Yes", "No"])
                run_report = st.form_submit_button("Generate Multi-Agent Audit Report")
            if run_report:
                try:
                    # Load models
                    pricing_model = joblib.load(os.path.join(MODELS_DIR, "best_pricing_model.joblib"))
                    scaler_pricing = joblib.load(os.path.join(MODELS_DIR, "scaler_pricing.joblib"))
                    le_orig = joblib.load(os.path.join(MODELS_DIR, "encoder_origin.joblib"))
                    le_dest = joblib.load(os.path.join(MODELS_DIR, "encoder_dest.joblib"))

                    delay_model = joblib.load(os.path.join(MODELS_DIR, "best_delay_model.joblib"))
                    scaler_delay = joblib.load(os.path.join(MODELS_DIR, "scaler_delay.joblib"))
                    le_delay_orig = joblib.load(os.path.join(MODELS_DIR, "encoder_delay_origin.joblib"))
                    le_delay_dest = joblib.load(os.path.join(MODELS_DIR, "encoder_delay_dest.joblib"))

                    compliance_model = joblib.load(os.path.join(MODELS_DIR, "best_compliance_model.joblib"))
                    scaler_compliance = joblib.load(os.path.join(MODELS_DIR, "scaler_compliance.joblib"))
                    le_carrier = joblib.load(os.path.join(MODELS_DIR, "encoder_compliance_carrier.joblib"))

                    distance_map = {
                        ("Shanghai", "JNPT Mumbai"): 5000, ("Shanghai", "Mundra"): 5200, ("Shanghai", "Chennai"): 4200, ("Shanghai", "Cochin"): 4500,
                        ("Rotterdam", "JNPT Mumbai"): 7200, ("Rotterdam", "Mundra"): 7000, ("Rotterdam", "Chennai"): 8200, ("Rotterdam", "Cochin"): 7900,
                        ("Singapore", "JNPT Mumbai"): 3000, ("Singapore", "Mundra"): 3300, ("Singapore", "Chennai"): 1800, ("Singapore", "Cochin"): 2200,
                        ("New York", "JNPT Mumbai"): 11500, ("New York", "Mundra"): 11200, ("New York", "Chennai"): 12000, ("New York", "Cochin"): 11800,
                    }
                    dist = distance_map.get((c_origin, c_destination), 5000)

                    o_enc = le_orig.transform([c_origin])[0]
                    d_enc = le_dest.transform([c_destination])[0]
                    p_feats = np.array([[dist, c_weight, c_congestion, c_fuel, c_rating, o_enc, d_enc]])
                    pred_price = pricing_model.predict(scaler_pricing.transform(p_feats))[0]

                    do_enc = le_delay_orig.transform([c_origin])[0]
                    dd_enc = le_delay_dest.transform([c_destination])[0]
                    d_feats = np.array([[dist, c_weather, c_congestion, c_ontime, 8.0, do_enc, dd_enc]])
                    delay_prob = delay_model.predict_proba(scaler_delay.transform(d_feats))[0][1]

                    car_enc = le_carrier.transform([c_carrier])[0]
                    m_val = 1 if c_maint == "Passed" else 0
                    i_val = 1 if c_ins == "Yes" else 0
                    c_feats = np.array([[c_ontime, c_violations, m_val, c_rating, i_val, car_enc]])
                    compliance_prob = compliance_model.predict_proba(scaler_compliance.transform(c_feats))[0][1]

                    st.session_state.predictions_count += 3
                    report_json = llm_engine.generate_audit_report(pred_price, delay_prob, compliance_prob, c_origin, c_destination, c_weight, c_carrier)

                    st.markdown("### 📄 Structured System Audit Report")
                    st.code(report_json, language="json")
                    report_obj = json.loads(report_json)
                    st.info(report_obj['overall_audit_summary'])
                except Exception as e: st.error(f"Report Error: {e}")

    elif st.session_state.page == "Settings":
        st.markdown("## Settings & Control")
        with st.container(border=True):
            st.markdown("### Model Pipelines")
            if st.button("Retrain All ML Agents", use_container_width=True):
                with st.spinner("Retraining..."): train_ml.train_all_agents()
                st.success("Retraining finished.")
                time.sleep(1)
                st.rerun()
        with st.container(border=True):
            st.markdown("### Database Housekeeping")
            if st.button("Flush Lockouts & OTP rate-limits", use_container_width=True):
                users = db.get_all_users()
                for u in users:
                    db.unlock_user(u[1])
                    db.reset_otp_cooldown(u[1])
                st.success("Lockouts reset.")
        with st.container(border=True):
            st.markdown("### System Telemetry")
            gpu_status = "🟢 ACTIVE (Using Quantized Qwen2.5 GPU)" if llm_engine.GPU_AVAILABLE else "🔴 INACTIVE (Using Rule-Based CPU fallback)"
            st.write(f"GPU Accelerator Status: **{gpu_status}**")

    elif st.session_state.page == "Admin":
        if role == "Admin": admin_dash.show_admin_dashboard()
        else:
            st.error("Access denied.")
            navigate("Home")
''')

# 9. Train all machine learning agents
print("\n--- Training ML Agents ---")
import train_ml_freight
train_ml_freight.train_all_agents()

# 10. Load Colab Secrets & Start ngrok Tunnel
print("\n--- Launching Streamlit & Ngrok ---")
try:
    from google.colab import userdata

    email_address = userdata.get('EMAIL_ADDRESS')
    email_password = userdata.get('EMAIL_PASSWORD')
    ngrok_token = userdata.get('NGROK_AUTHTOKEN')
    jwt_secret = userdata.get('JWT_SECRET')
    hf_token = userdata.get('HF_TOKEN')

    if email_address: os.environ['EMAIL_ADDRESS'] = email_address
    if email_password: os.environ['EMAIL_PASSWORD'] = email_password
    if jwt_secret: os.environ['JWT_SECRET'] = jwt_secret
    if hf_token: os.environ['HF_TOKEN'] = hf_token

    if ngrok_token:
        from pyngrok import ngrok
        ngrok.set_auth_token(ngrok_token)
        print("✅ NGROK_AUTHTOKEN registered successfully.")
    else:
        print("⚠️ Warning: NGROK_AUTHTOKEN secret not found.")
except Exception as e:
    print(f"⚠️ Secrets/Environment Notice: {e}")

# Kill any existing streamlit sessions
subprocess.run(["pkill", "-f", "streamlit"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(1)

# Start Streamlit
print("🚀 Starting Streamlit server...")
env = os.environ.copy()
env["STREAMLIT_SERVER_FILE_WATCHER_TYPE"] = "none"
process = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501"],
    env=env,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
time.sleep(4) # Wait for server startup

# Expose Streamlit via Ngrok
try:
    from pyngrok import ngrok
    tunnels = ngrok.get_tunnels()
    if tunnels:
        public_url = tunnels[0].public_url
        print("♻️ Reusing existing active Ngrok tunnel...")
    else:
        public_url = ngrok.connect(8501).public_url
        print("🚀 Created new Ngrok tunnel...")

    print("\n" + "=" * 65)
    print(f"🚀 Streamlit Freight Portal Live: {public_url}")
    print("=" * 65)
    print("⏳ Keep this cell running to maintain connection. Click Stop to close.")

    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("\n🛑 Shutting down server...")
    from pyngrok import ngrok
    ngrok.kill()
    process.terminate()
    print("✅ Stopped.")
except Exception as e:
    print(f"❌ Ngrok failed: {e}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 102.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 127.9 MB/s eta 0:00:00

--- Training ML Agents ---

--- Training Agent 1: Dynamic Pricing (Regression) ---
-> RandomForestRegressor | R2: 0.9909 | RMSE: 226.5236
-> GradientBoostingRegressor | R2: 0.9959 | RMSE: 152.4749
-> ExtraTreesRegressor | R2: 0.9947 | RMSE: 171.9653
-> RidgeRegression | R2: 0.9998 | RMSE: 29.8844
-> DecisionTreeRegressor | R2: 0.9775 | RMSE: 355.4286
-> AdaBoostRegressor | R2: 0.9635 | RMSE: 453.0570
-> KNeighborsRegressor | R2: 0.9717 | RMSE: 398.6337
[Best Pricing Model] RidgeRegression with R2: 0.9998

--- Training Agent 2: Route Delay Prediction (Classification) ---
-> RandomForestClassifier | ROC-AUC: 0.9856 | Accuracy: 0.9300
-> GradientBoostingClassifier | ROC-AUC